# Micro-Expression Spotting & Classification (CASME II)

In [1]:
import os
import sys

def _find_project_root(marker="pyproject.toml", max_up=8):
    path = os.path.abspath(os.getcwd())
    for _ in range(max_up):
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return os.path.abspath(os.getcwd())

project_root = _find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from article.src import MELabel, FrameSequence, MESpotting, RealTimeProfiler
from src.dataset.modules.behavioral_features import BehavioralFeatures
from src.models.modules.spatio_temporal.spatio_temporal_cnn import SpatioTemporalCNN

## 1. Load CASME II annotations & initialize utilities

In [2]:
casme2_dir = '/home/inadio/datasets/secondaries/casme-ii' if os.path.exists('/home/inadio/datasets/secondaries/casme-ii') else '/home/inadio/datasets/secondaries/casme-2'
annotations_path = os.path.join(casme2_dir, 'annotations.xlsx')
cache_dir = os.path.join(casme2_dir, 'cache')

df = pd.read_excel(annotations_path)
df = df[(df['OffsetFrame'] - df['OnsetFrame'] + 1) <= 100].copy()
print(f"Loaded and filtered annotations to {len(df)} micro-expression entries.")

FPS = 200
spotter = MESpotting(cutoff_ratio=0.20, fps=FPS, smooth_window_ms=300, thresh_window_ms=900)
extractor = BehavioralFeatures()
profiler = RealTimeProfiler()

sequences: list[FrameSequence] = []
clipped_sequences: list[FrameSequence] = []
all_features = []
labels = []
groups = []
spotted_intervals = []
gt_intervals = []

Loaded and filtered annotations to 244 micro-expression entries.


W0000 00:00:1788744286.555391   54903 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788744286.560174   56007 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788744286.573486   56009 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 2. Extract sequences, spot boundaries & slice micro-expressions

In [3]:
for _, row in df.iterrows():
    subject = row['Subject']
    subject_str = f"sub{int(subject):02d}" if pd.notnull(subject) else "sub01"
    filename = str(row['Filename'])
    raw_emotion = row['Estimated Emotion']

    label = MELabel.map(raw_emotion, target='2-class')
    if label is None:
        continue

    npz_path = os.path.join(cache_dir, f"{subject_str}_{filename}.npz")
    seq = FrameSequence.from_npz(npz_path, subject=subject_str, clip_name=filename, label=label, fps=FPS)
    if seq is None or len(seq) == 0:
        continue

    gt_interval = (int(row['OnsetFrame']), int(row['OffsetFrame']))

    # Best-IoU match against gt (standard detection-eval assignment), not
    # spot()'s arbitrary first-detected-candidate selection.
    candidates = spotter.spot_all(seq.magnitudes)
    match_result = spotter.match(candidates, gt_interval, fallback_mags=seq.magnitudes)
    feat_interval = match_result.get('feat_interval', gt_interval)
    spot_interval = match_result.get('spot_interval', gt_interval)

    clipped_seq = seq.clip(feat_interval[0], feat_interval[1])
    if len(clipped_seq) == 0:
        continue

    sequences.append(seq)
    clipped_sequences.append(clipped_seq)
    spotted_intervals.append(spot_interval)
    gt_intervals.append(gt_interval)
    labels.append(label)
    groups.append(subject_str)

    flow_tensor = torch.from_numpy(clipped_seq.flow)
    feats = extractor._extract(flow_tensor).cpu().numpy()
    all_features.append(feats)

## 3. Spotting Performance Benchmark (Fang et al. 2023 protocol)

In [4]:
spot_metrics = spotter.benchmark(spotted_intervals, gt_intervals, iou_thresh=0.5)
print("\n=== Spotting Performance (CASME II | Fang et al. 2023 Standard) ===")
print(f"Total Samples:       {spot_metrics['samples']}")
print(f"True Positives:      {spot_metrics['tp']} (IoU >= 0.5)")
print(f"Average IoU:         {spot_metrics['mean_iou']:.4f}")
print(f"Spotting Precision:  {spot_metrics['precision']:.4f}")
print(f"Spotting Recall:     {spot_metrics['recall']:.4f}")
print(f"Spotting F1-Score:   {spot_metrics['f1']:.4f}")
print("===================================================================\n")


=== Spotting Performance (CASME II | Fang et al. 2023 Standard) ===
Total Samples:       119
True Positives:      37 (IoU >= 0.5)
Average IoU:         0.3391
Spotting Precision:  0.3109
Spotting Recall:     0.3109
Spotting F1-Score:   0.3109



## 4. Feature Matrix & LOSO Cross-Validation (Baseline SVM)

In [5]:
y, class_names = MELabel.encode(labels)
groups_arr = np.array(groups)
X_static = np.stack([np.concatenate([f.mean(axis=0), f.std(axis=0)]) for f in all_features])

logo = LeaveOneGroupOut()
splits = list(logo.split(X_static, y, groups=groups_arr))
print(f"LOSO Cross-Validation | Total Subjects (Splits): {len(splits)}")

svm_preds = np.zeros_like(y)
for train_idx, test_idx in splits:
    X_train, y_train = X_static[train_idx], y[train_idx]
    X_test, y_test = X_static[test_idx], y[test_idx]

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', random_state=42)
    clf.fit(X_train_scaled, y_train)
    svm_preds[test_idx] = clf.predict(X_test_scaled)

print("\n=== CASME II SVM Classification Performance (LOSO) ===")
print(f"Accuracy:        {accuracy_score(y, svm_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, svm_preds, average='macro'):.4f}")
print(f"Macro Precision: {precision_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, svm_preds, average='macro', zero_division=0):.4f}")
print("=======================================================")

LOSO Cross-Validation | Total Subjects (Splits): 25

=== CASME II SVM Classification Performance (LOSO) ===
Accuracy:        0.6050
Macro F1-Score:  0.4135
Macro Precision: 0.4026
Macro Recall:    0.4335


## 5. SpatioTemporalCNN 3D Deep Learning Classification (LOSO)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nEvaluating SpatioTemporalCNN 3D Deep Learning model on device: {device}...")

cnn_preds = np.zeros_like(y)
batch_size = 8

for fold_idx, (train_idx, test_idx) in enumerate(splits):
    train_seqs = [clipped_sequences[i] for i in train_idx]
    test_seqs = [clipped_sequences[i] for i in test_idx]
    y_train = torch.tensor(y[train_idx], dtype=torch.long, device=device)

    model = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names), dropout_p=0.3).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(10):
        perm = torch.randperm(len(train_seqs))
        for i in range(0, len(train_seqs), batch_size):
            indices = perm[i : i + batch_size]
            if len(indices) < 2:
                continue
            batch_seqs = [train_seqs[idx] for idx in indices]
            batch_x = FrameSequence.pad_batch(batch_seqs, max_len=64, device=device)
            batch_y = y_train[indices]

            optimizer.zero_grad()
            out = model(batch_x)
            loss = criterion(out, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_fold_preds = []
        for i in range(0, len(test_seqs), batch_size):
            batch_seqs = test_seqs[i : i + batch_size]
            batch_x = FrameSequence.pad_batch(batch_seqs, max_len=64, device=device)
            out_test = model(batch_x)
            test_fold_preds.append(torch.argmax(out_test, dim=1).cpu().numpy())
        if test_fold_preds:
            cnn_preds[test_idx] = np.concatenate(test_fold_preds)

print("\n=== CASME II SpatioTemporalCNN Classification Performance (LOSO) ===")
print(f"Accuracy:        {accuracy_score(y, cnn_preds):.4f}")
print(f"Macro F1-Score:  {f1_score(y, cnn_preds, average='macro'):.4f}")
print(f"Macro Precision: {precision_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall:    {recall_score(y, cnn_preds, average='macro', zero_division=0):.4f}")
print("====================================================================")
print("Classification Report:")
print(classification_report(y, cnn_preds, target_names=class_names, zero_division=0))


Evaluating SpatioTemporalCNN 3D Deep Learning model on device: cuda...

=== CASME II SpatioTemporalCNN Classification Performance (LOSO) ===
Accuracy:        0.6975
Macro F1-Score:  0.4796
Macro Precision: 0.5170
Macro Recall:    0.5066
Classification Report:
              precision    recall  f1-score   support

    negative       0.73      0.92      0.82        87
    positive       0.30      0.09      0.14        32

    accuracy                           0.70       119
   macro avg       0.52      0.51      0.48       119
weighted avg       0.62      0.70      0.64       119



## 6. Real-Time End-to-End Latency Benchmarking with RealTimeProfiler

In [7]:
print("\n=== Running Real-Time Performance Profiling ===")
scaler_full = RobustScaler().fit(X_static)
svm_full = SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', random_state=42).fit(scaler_full.transform(X_static), y)

cnn_full = SpatioTemporalCNN(in_channels=10, num_classes=len(class_names)).to(device)
cnn_full.eval()

frame_counts = [len(s) for s in sequences]

for seq in sequences:
    with profiler.record("spot"):
        feat_int, _ = spotter.spot(seq.magnitudes, fallback_half_win=49)

    clipped = seq.clip(feat_int[0], feat_int[1])
    if len(clipped) == 0:
        continue

    with profiler.record("cnn_infer"):
        x_cnn = clipped.to_tensor(device=device)
        with torch.no_grad():
            _ = cnn_full(x_cnn)

print("\nReal-Time Performance Statistics:")
print(profiler.report(frame_counts).to_string(index=False))


=== Running Real-Time Performance Profiling ===

Real-Time Performance Statistics:
                 Metric            Value
          Latency: Spot   0.794 ms / seq
     Latency: Cnn_infer   2.517 ms / seq
 Total Sequence Latency         3.311 ms
Average Sequence Length     261.0 frames
Estimated Frame Latency 0.013 ms / frame
       Throughput (FPS)      78834.2 FPS
